# Moment 4: distance_gradient_slope

The lightweight one -- a documentation and unit-reconciliation task, not a microdata computation.
No survey in `Data/` reports employment binned by kilometre-distance from a CBD (confirmed during
the literature-verification pass, `paper/notes/literature-verification.md` section 5); the only
usable source is Baez and Kshirsagar (2026), World Bank Policy Research Working Paper 11285,
Table 5b.

## The unit mismatch this moment already survived once

`DECISIONS.md` records this as a locked moment that was originally sourced to the *wrong* table:
Baez and Kshirsagar's Table 1 (population density by kilometre band) was cited in early drafts as
"employment rate by distance band," but their actual employment result -- Table 5b -- is a
continuous coefficient of employment share on a **within-city, population-weighted percentile
rank of network travel time**, not a km-distance gradient. D6 already settled *how* this paper is
used (digitised from the published table, not reproduced from `RR_ZAF_2025_490`, which needs
licensed Stata 18 MP); this notebook settles *which number*, since the source reports three
separate city-specific coefficients, not one.

## Picking a value: mean across cities, not one metro

Table 5b (p.38 printed) reports a 10-percentile increase in travel-time rank corresponding to a
drop in employment share of 4.9 percentage points in Johannesburg, 3.7 in Cape Town, and 6.5 in
eThekwini. The model is a single stylised city, not any one of these three specific metros, so
picking one city over the other two would be an arbitrary choice the model has no basis for --
the mean across all three, 5.03 pp per 10 percentile points, is used instead.

## The "standard error" here is cross-city spread, not a sampling SE, and is labelled as such

The source paper's own standard errors for these coefficients weren't recoverable -- the working
paper PDF isn't text-extractable in this environment (no PDF-rendering tool available), and no
other accessible source reports them. Rather than leave `standard_error` blank or invent a number
that looks like a formal statistic, the sample standard deviation across the three cities'
point estimates (1.40) is used as an explicit uncertainty proxy: it measures genuine city-to-city
heterogeneity in this coefficient, not the sampling uncertainty of any single estimate, and both
`moments.csv`'s `source` column and this cell say so plainly rather than let the number pass as
something it isn't."

In [ ]:
import statistics
from pathlib import Path

import pandas as pd

# Baez and Kshirsagar (2026), WB WP 11285, Table 5b (p.38 printed): percentage-point drop in
# employment share per 10-percentile increase in within-city travel-time rank to the nearest
# business district.
CITY_COEFFICIENTS = {
    "Johannesburg": 4.9,
    "Cape Town": 3.7,
    "eThekwini": 6.5,
}

mean_slope = statistics.mean(CITY_COEFFICIENTS.values())
cross_city_sd = statistics.stdev(CITY_COEFFICIENTS.values())

print("city coefficients:", CITY_COEFFICIENTS)
print(f"mean: {mean_slope:.4f}")
print(f"cross-city sample SD (uncertainty proxy, not a sampling SE): {cross_city_sd:.4f}")

In [ ]:
moments_path = Path("../../data/moments.csv")
moments = pd.read_csv(moments_path)
moments["period"] = moments["period"].astype("object")
moments["source"] = moments["source"].astype("object")
row = moments["key"] == "distance_gradient_slope"
moments.loc[row, "value"] = round(mean_slope, 4)
moments.loc[row, "standard_error"] = round(cross_city_sd, 4)
moments.loc[row, "period"] = "2026"
moments.loc[row, "source"] = (
    "Baez, J. and Kshirsagar, V. (2026). South Africa's Fragmented Cities: The Unequal "
    "Burden of Labor Market Frictions. World Bank Policy Research Working Paper 11285. "
    "Table 5b (p.38 printed) -- mean of Johannesburg (4.9), Cape Town (3.7) and eThekwini "
    "(6.5) pp employment-share drop per 10-percentile increase in travel-time rank. "
    "standard_error is the cross-city sample SD, not a reported sampling SE -- see notebook 04."
)
moments.loc[row, "provisional"] = False
moments.to_csv(moments_path, index=False)
moments[row]